In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets", "openpyxl", "tqdm"])

import os, random, time, copy
import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR = "/kaggle/working/"; os.makedirs(OUTPUT_DIR, exist_ok=True)

D = 256; VOCAB_SIZE = 30522; MAX_SEQ = 64; HEADS = 4; DROP = 0.25
CBAM_BLOCKS = 3; TEXT_ENC_LAYERS = 2; TEXT_REFINE_LAYERS = 2; FUSE_LAYERS = 4
NUM_CLIENTS = 5; ROUNDS = 20; LOCAL_EP = 3; BS = 32; FED_LR = 1e-4; WD = 1e-4


# ─────────────────────────────────────────────────────────────────────
# 1. VQA-RAD  (flaviagiammarino/vqa-rad)
# ─────────────────────────────────────────────────────────────────────
# ~2,248 QA pairs · 315 radiology images
# Heavy yes/no split (~50%), plus open-ended: modality, organ, abnormality,
# plane, counting, color, size, attribute, positional reasoning.
 
def normalize_answer_vqarad(ans: str) -> str:
    """Normalize VQA-RAD answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)   # remove stray punctuation
    ans = re.sub(r'\s+', ' ', ans).strip()    # collapse whitespace
 
    # ── Yes / No canonicalization ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true',
               'yes it is', 'yes there is', 'yes, it is', 'yes, there is',
               'affirmative', 'positive', 'right'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'no, it is not', 'no it is not', 'no, there is not',
               'no there is not', 'none', 'not present', 'no, there isnt'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'
 
    # ── Numeric canonicalization ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]
 
    # ── Modality synonyms ──
    modality_map = {
        'ct scan': 'ct', 'cat scan': 'ct', 'computed tomography': 'ct',
        'ct - Loss contrast': 'ct',
        'magnetic resonance imaging': 'mri', 'mr': 'mri', 'mr imaging': 'mri',
        'mri - Loss contrast': 'mri', 'mri - flair': 'mri',
        't1': 'mri', 't2': 'mri', 'flair': 'mri', 'dwi': 'mri',
        'x-ray': 'xray', 'x ray': 'xray', 'xr': 'xray',
        'plain film': 'xray', 'radiograph': 'xray',
        'chest x-ray': 'xray', 'chest x ray': 'xray', 'cxr': 'xray',
        'pa': 'xray', 'ap': 'xray',
        'ultrasound': 'ultrasound', 'us': 'ultrasound', 'sonography': 'ultrasound',
        'pet scan': 'pet', 'positron emission tomography': 'pet',
        'mammogram': 'mammography', 'mammo': 'mammography',
    }
    if ans in modality_map:
        return modality_map[ans]
 
    # ── Plane / orientation synonyms ──
    plane_map = {
        'axial view': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'coronal view': 'coronal', 'frontal': 'coronal',
        'sagittal view': 'sagittal', 'lateral': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]
 
    # ── Anatomical synonyms ──
    anatomy_map = {
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'lungs': 'lung', 'pulmonary': 'lung',
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain', 'intracranial': 'brain',
        'abdomen': 'abdomen', 'abdominal': 'abdomen', 'belly': 'abdomen',
        'stomach': 'abdomen',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'heart': 'heart', 'cardiac': 'heart',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'skull': 'skull', 'calvarium': 'skull',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]
 
    # ── Laterality ──
    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both': 'bilateral', 'both sides': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]
 
    # ── Remove trailing articles / fillers ──
    ans = re.sub(r'\bthe\b', '', ans).strip()
    ans = re.sub(r'\ba\b', '', ans).strip()
    ans = re.sub(r'\s+', ' ', ans)
 
    return ans

# =============================================================================
# LOAD DATASET INTO RAM
# =============================================================================
print("\n" + "="*60 + "\nLOADING VQA-RAD INTO RAM\n" + "="*60)
from datasets import load_dataset
#ds = load_dataset('mdwiratathya/SLAKE-vqa-english')
ds = load_dataset('flaviagiammarino/vqa-rad')
#ds = load_dataset('flaviagiammarino/path-vqa')

def extract(sd, name):
    samples = []
    for s in tqdm(sd, desc=name):
        try:
            img = s.get('image'); q = str(s.get('question','')); a = str(s.get('answer','')).strip().lower()
            a = normalize_answer_vqarad(a)
            if img and q and a:
                samples.append({'image': np.array(img.convert('RGB').resize((224,224)), dtype=np.float32)/255.0, 'question': q, 'answer': a})
        except: continue
    print(f"  {name}: {len(samples)}"); return samples
train_samples = extract(ds['train'], 'train'); test_samples = extract(ds['test'], 'test'); del ds
all_ans = [s['answer'] for s in train_samples + test_samples]
answer_vocab = {'<unk>': 0}
for i, a in enumerate(sorted(set(all_ans))): answer_vocab[a] = i + 1
num_classes = len(answer_vocab)
print(f"  Vocab: {num_classes}, Train: {len(train_samples)}, Test: {len(test_samples)}")

def tokenize(questions):
    ids_l, mask_l = [], []
    for q in questions:
        w = q.lower().split()[:MAX_SEQ-2]
        ids = [1] + [hash(x)%(VOCAB_SIZE-2)+2 for x in w] + [2]
        m = [1.0]*len(ids)
        while len(ids) < MAX_SEQ: ids.append(0); m.append(0.0)
        ids_l.append(ids[:MAX_SEQ]); mask_l.append(m[:MAX_SEQ])
    return ids_l, mask_l

class VQADataset(Dataset):
    def __init__(self, samples, vocab, augment=False):
        self.samples=samples; self.vocab=vocab; self.augment=augment
        self.ids, self.masks = tokenize([s['question'] for s in samples])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]; img = torch.tensor(s['image']).permute(2,0,1)
        if self.augment and random.random()>0.5: img = img.flip(-1)
        return img, torch.tensor(self.ids[idx], dtype=torch.long), torch.tensor(self.masks[idx], dtype=torch.float32), self.vocab.get(s['answer'],0)

def collate_fn(batch):
    imgs, ids, masks, lbls = zip(*batch)
    return torch.stack(imgs), torch.stack(ids), torch.stack(masks), torch.tensor(lbls, dtype=torch.long)

idx = np.random.permutation(len(train_samples)); sz = len(train_samples)//NUM_CLIENTS
client_splits = {c: idx[c*sz:(c+1)*sz if c<NUM_CLIENTS-1 else len(train_samples)].tolist() for c in range(NUM_CLIENTS)}
client_loaders, client_sizes = {}, {}
for cid, indices in client_splits.items():
    client_loaders[cid] = DataLoader(VQADataset([train_samples[i] for i in indices], answer_vocab, augment=True),
                                      batch_size=BS, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
    client_sizes[cid] = len(indices); print(f"  Client {cid}: {len(indices)} samples")
test_loader = DataLoader(VQADataset(test_samples, answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

# =============================================================================
# MODEL — SAME as centralized (copy-paste identical)
# =============================================================================
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads=4, ffn_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1=nn.LayerNorm(dim); self.norm2=nn.LayerNorm(dim)
        self.attn=nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.ffn=nn.Sequential(nn.Linear(dim,dim*ffn_ratio), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*ffn_ratio,dim), nn.Dropout(dropout))
    def forward(self, x, mask=None):
        h=self.norm1(x); kpm=(mask==0) if mask is not None else None
        h,_=self.attn(h,h,h,key_padding_mask=kpm); x=x+h; return x+self.ffn(self.norm2(x))

class VisionEncoder(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.conv1=nn.Conv2d(3,32,7,stride=2,padding=3,bias=False); self.bn1=nn.BatchNorm2d(32)
        self.pool1=nn.MaxPool2d(3,stride=2,padding=1)
        self.conv2=nn.Conv2d(32,64,3,stride=2,padding=1,bias=False); self.bn2=nn.BatchNorm2d(64)
        self.conv3=nn.Conv2d(64,128,3,stride=2,padding=1,bias=False); self.bn3=nn.BatchNorm2d(128)
        self.conv4=nn.Conv2d(128,dim,3,stride=2,padding=1,bias=False); self.bn4=nn.BatchNorm2d(dim)
        self.norm=nn.LayerNorm(dim)
    def forward(self, x):
        h=self.pool1(F.silu(self.bn1(self.conv1(x)))); h=F.silu(self.bn2(self.conv2(h)))
        h=F.silu(self.bn3(self.conv3(h))); h=F.silu(self.bn4(self.conv4(h)))
        B,C,H,W=h.shape; return self.norm(h.permute(0,2,3,1).reshape(B,H*W,C))

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=30522, dim=256, n_layers=2, n_heads=4, max_len=64, dropout=0.1):
        super().__init__()
        self.tok_embed=nn.Embedding(vocab_size,dim); self.pos_embed=nn.Parameter(torch.randn(1,max_len,dim)*0.02)
        self.embed_norm=nn.LayerNorm(dim); self.embed_drop=nn.Dropout(dropout)
        self.blocks=nn.ModuleList([TransformerBlock(dim,n_heads,dropout=dropout) for _ in range(n_layers)])
        self.final_norm=nn.LayerNorm(dim)
    def forward(self, input_ids, mask=None):
        L=input_ids.shape[1]; x=self.tok_embed(input_ids)+self.pos_embed[:,:L,:]
        x=self.embed_drop(self.embed_norm(x))
        for blk in self.blocks: x=blk(x,mask=mask)
        return self.final_norm(x)

class ChannelAttention(nn.Module):
    def __init__(self, ch, ratio=8):
        super().__init__(); self.fc1=nn.Linear(ch,ch//ratio,bias=False); self.fc2=nn.Linear(ch//ratio,ch,bias=False)
    def forward(self, x):
        avg=x.mean(dim=[1,2],keepdim=True); mx=x.amax(dim=[1,2],keepdim=True)
        return x*torch.sigmoid(self.fc2(F.silu(self.fc1(avg)))+self.fc2(F.silu(self.fc1(mx))))

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__(); self.conv1=nn.Conv2d(2,8,3,padding=1,bias=False); self.conv2=nn.Conv2d(2,8,3,padding=2,dilation=2,bias=False)
        self.fuse=nn.Conv2d(16,1,1,bias=False)
    def forward(self, x):
        xp=x.permute(0,3,1,2); avg=xp.mean(1,keepdim=True); mx=xp.amax(1,keepdim=True)
        cat=torch.cat([avg,mx],1); ms=torch.cat([self.conv1(cat),self.conv2(cat)],1)
        return x*torch.sigmoid(self.fuse(ms)).permute(0,2,3,1)

class CBAMBlock(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.ca=ChannelAttention(ch); self.sa=SpatialAttention()
        self.ffn=nn.Sequential(nn.Linear(ch,ch*2),nn.GELU(),nn.Linear(ch*2,ch))
        self.norm1=nn.LayerNorm(ch); self.norm2=nn.LayerNorm(ch)
    def forward(self, tokens):
        B,N,C=tokens.shape; sp=tokens.reshape(B,7,7,C); sp=self.sa(self.ca(sp))
        tokens=self.norm1(tokens+sp.reshape(B,N,C)); return self.norm2(tokens+self.ffn(tokens))

class FusionLayer(nn.Module):
    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        self.v2t=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); self.v2t_norm=nn.LayerNorm(dim)
        self.v2t_ffn=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*4,dim)); self.v2t_ffn_norm=nn.LayerNorm(dim)
        self.t2v=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); self.t2v_norm=nn.LayerNorm(dim)
        self.t2v_ffn=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*4,dim)); self.t2v_ffn_norm=nn.LayerNorm(dim)
    def forward(self, v, t, text_kpm=None):
        o,_=self.v2t(v,t,t,key_padding_mask=text_kpm); v=self.v2t_norm(v+o); v=self.v2t_ffn_norm(v+self.v2t_ffn(v))
        o,_=self.t2v(t,v,v); t=self.t2v_norm(t+o); t=self.t2v_ffn_norm(t+self.t2v_ffn(t)); return v,t

class MedicalVQAModel(nn.Module):
    """SAME model as centralized."""
    def __init__(self, num_classes):
        super().__init__()
        self.vision_enc=VisionEncoder(D); self.text_enc=TextEncoder(VOCAB_SIZE,D,TEXT_ENC_LAYERS,HEADS,MAX_SEQ,DROP)
        self.vis_refine=nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)])
        self.text_refine=nn.ModuleList([TransformerBlock(D,HEADS,dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)])
        self.text_refine_norm=nn.LayerNorm(D)
        self.q_attn=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); self.q_gate=nn.Linear(D,D); self.q_norm=nn.LayerNorm(D)
        self.fusion_layers=nn.ModuleList([FusionLayer(D,HEADS,DROP) for _ in range(FUSE_LAYERS)])
        self.pool_query=nn.Parameter(torch.randn(1,1,D)*0.02)
        self.pool_attn=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); self.pool_norm=nn.LayerNorm(D)
        self.head_fc1=nn.Linear(D,D); self.head_drop1=nn.Dropout(DROP)
        self.head_fc2=nn.Linear(D,D//2); self.head_drop2=nn.Dropout(DROP)
        self.head_out=nn.Linear(D//2,num_classes); self.head_res=nn.Linear(D,D//2); self.head_norm=nn.LayerNorm(D//2)
    def forward(self, images, input_ids, text_mask):
        v=self.vision_enc(images); t=self.text_enc(input_ids,mask=text_mask)
        for blk in self.vis_refine: v=blk(v)
        for blk in self.text_refine: t=blk(t,mask=text_mask)
        t=self.text_refine_norm(t)
        q_cls=t[:,0:1,:].expand(-1,v.shape[1],-1); ao,_=self.q_attn(q_cls,v,v)
        gate=torch.sigmoid(self.q_gate(ao)); v=self.q_norm(v+v*gate+ao*(1-gate))
        kpm=(text_mask==0)
        for fl in self.fusion_layers: v,t=fl(v,t,text_kpm=kpm)
        combined=torch.cat([v,t],dim=1); B=combined.shape[0]; pq=self.pool_query.expand(B,-1,-1)
        pooled,_=self.pool_attn(pq,combined,combined); fused=self.pool_norm(pq+pooled).squeeze(1)
        h=self.head_drop1(F.gelu(self.head_fc1(fused))); h=self.head_drop2(F.gelu(self.head_fc2(h)))
        return self.head_out(self.head_norm(h+self.head_res(fused)))

global_model = MedicalVQAModel(num_classes).to(device)
n_params = sum(p.numel() for p in global_model.parameters())
print(f"\n  Model: {n_params:,} ({n_params/1e6:.1f}M)")

# =============================================================================
# FedAvg TRAINING
# =============================================================================
print("\n" + "="*60 + "\nTRAINING (FedAvg)\n" + "="*60)

def get_params(m): return [p.data.cpu().numpy().copy() for p in m.parameters()]
def set_params(m, params):
    for p, w in zip(m.parameters(), params): p.data = torch.from_numpy(w).to(p.device)
def fedavg_agg(cp, sizes):
    total=sum(sizes); wts=[n/total for n in sizes]
    return [sum(wts[i]*cp[i][p] for i in range(len(cp))) for p in range(len(cp[0]))]

criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def eval_model(loader):
    global_model.eval(); ls,c,t = 0.0,0,0
    for imgs,ids,masks,lbls in loader:
        imgs,ids,masks,lbls = imgs.to(device),ids.to(device),masks.to(device),lbls.to(device)
        logits=global_model(imgs,ids,masks); loss=criterion(logits,lbls)
        ls+=loss.item()*lbls.size(0); c+=(logits.argmax(-1)==lbls).sum().item(); t+=lbls.size(0)
    return ls/t, 100*c/t

history = {'round':[],'avg_train_loss':[],'avg_train_acc':[],'test_loss':[],'test_acc':[],'round_time':[]}
# ── ADDED TIME HISTORY INITIALIZATION ──
for cid in range(NUM_CLIENTS):
    history[f'c{cid}_loss']=[]; history[f'c{cid}_acc']=[]; history[f'c{cid}_time']=[]
    
best_acc, best_state = 0.0, None

for rnd in range(1, ROUNDS+1):
    t0=time.time(); gp=get_params(global_model); round_cp=[]; rl=[]; rc,rt=0,0
    pbar=tqdm(range(NUM_CLIENTS), desc=f"R{rnd:02d}/{ROUNDS}", leave=False)
    for cid in pbar:
        c_t0 = time.time() # ── START CLIENT TIMER ──
        
        local=copy.deepcopy(global_model); set_params(local,[p.copy() for p in gp]); local.train()
        opt=torch.optim.AdamW(local.parameters(), lr=FED_LR, weight_decay=WD)
        cl,cc,ct=0.0,0,0
        for _ in range(LOCAL_EP):
            for imgs,ids,masks,lbls in client_loaders[cid]:
                imgs,ids,masks,lbls=imgs.to(device),ids.to(device),masks.to(device),lbls.to(device)
                opt.zero_grad(); logits=local(imgs,ids,masks); loss=criterion(logits,lbls)
                loss.backward(); nn.utils.clip_grad_norm_(local.parameters(),1.0); opt.step()
                cl+=loss.item()*lbls.size(0); cc+=(logits.argmax(-1)==lbls).sum().item(); ct+=lbls.size(0)
                
        c_time = time.time() - c_t0 # ── END CLIENT TIMER ──
        
        round_cp.append(get_params(local))
        c_l=cl/max(ct,1); c_a=100*cc/max(ct,1); rl.append(c_l); rc+=cc; rt+=ct
        history[f'c{cid}_loss'].append(round(c_l,4)); history[f'c{cid}_acc'].append(round(c_a,2))
        history[f'c{cid}_time'].append(round(c_time,2)) # ── SAVE TO HISTORY ──
        
        # ── ADDED TIME TO TQDM POSTFIX ──
        pbar.set_postfix(C=cid, l=f"{c_l:.3f}", a=f"{c_a:.1f}%", t=f"{c_time:.1f}s"); del local,opt

    set_params(global_model, fedavg_agg(round_cp, list(client_sizes.values())))
    te_l,te_a=eval_model(test_loader); rtime=time.time()-t0
    avg_l=np.mean(rl); avg_a=100*rc/max(rt,1)
    history['round'].append(rnd); history['avg_train_loss'].append(round(avg_l,4))
    history['avg_train_acc'].append(round(avg_a,2)); history['test_loss'].append(round(te_l,4))
    history['test_acc'].append(round(te_a,2)); history['round_time'].append(round(rtime,1))
    mk=""
    if te_a>best_acc: best_acc=te_a; best_state=copy.deepcopy(global_model.state_dict()); mk=" ★"
    print(f"R{rnd:02d} [{rtime:.1f}s]  Train: {avg_l:.4f}/{avg_a:.1f}%  Test: {te_l:.4f}/{te_a:.1f}%{mk}")

if best_state: global_model.load_state_dict(best_state)
te_l,te_a=eval_model(test_loader)
print(f"\n{'='*60}\nFINAL: {te_a:.2f}%\n{'='*60}")

# =============================================================================
# SAVE EXCEL
# =============================================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
wb=openpyxl.Workbook(); ws=wb.active; ws.title="Training"
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='1A5276',end_color='1A5276',fill_type='solid')

# ── ADDED TIME TO HEADERS ──
headers=['Round','Avg Train Loss','Avg Train Acc (%)','Test Loss','Test Acc (%)','Time (s)']
for cid in range(NUM_CLIENTS): headers+=[f'C{cid} Loss',f'C{cid} Acc (%)', f'C{cid} Time (s)']

for c,h in enumerate(headers,1): cl=ws.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')

for i,rnd in enumerate(history['round']):
    r=i+2; ws.cell(row=r,column=1,value=rnd); ws.cell(row=r,column=2,value=history['avg_train_loss'][i])
    ws.cell(row=r,column=3,value=history['avg_train_acc'][i]); ws.cell(row=r,column=4,value=history['test_loss'][i])
    ws.cell(row=r,column=5,value=history['test_acc'][i]); ws.cell(row=r,column=6,value=history['round_time'][i])
    
    # ── MODIFIED COLUMN OFFSETS TO INCLUDE TIME (3 columns per client) ──
    for cid in range(NUM_CLIENTS):
        ws.cell(row=r,column=7+cid*3,value=history[f'c{cid}_loss'][i])
        ws.cell(row=r,column=8+cid*3,value=history[f'c{cid}_acc'][i])
        ws.cell(row=r,column=9+cid*3,value=history[f'c{cid}_time'][i])

ws2=wb.create_sheet("Summary")
for i,(k,v) in enumerate([("Method","FedAvg"),("Dataset","VQA-RAD"),("Params",f"{n_params:,}"),("Clients",NUM_CLIENTS),
    ("Local Epochs",LOCAL_EP),("Rounds",ROUNDS),("Best Test",round(best_acc,2)),("Final Test",round(te_a,2))],1):
    ws2.cell(row=i,column=1,value=k).font=Font(bold=True,name='Arial'); ws2.cell(row=i,column=2,value=v)
for s in [ws,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2
p=f"{OUTPUT_DIR}/vqarad_fedavg_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")

Device: cuda
GPU: Tesla T4, VRAM: 15.6 GB

LOADING VQA-RAD INTO RAM


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-eb8844602202be(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e5bc3d208bb4dee(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1793 [00:00<?, ? examples/s]

...

R18 [21.4s]  Train: 1.2590/87.2%  Test: 3.4290/36.4%


R19/20:   0%|          | 0/5 [00:00<?, ?it/s]

R19 [21.1s]  Train: 1.2076/88.1%  Test: 3.3877/36.8%


R20/20:   0%|          | 0/5 [00:00<?, ?it/s]

R20 [21.2s]  Train: 1.1452/89.4%  Test: 3.3901/36.4%

FINAL: 36.81%

Saved → /kaggle/working//vqarad_fedavg_results.xlsx
DONE!
